# Improving and evaluating prompts

### Building effective prompts and testing how they perform

Prompt Engineering -> A set of best practices and guidance to **improve** your prompts

Prompt Evaluation -> Automated testing to **measure** how well your prompts work

---

## Prompt Evaluation Workflow

- Many different ways to assemble a workflow
- Many different open source and paid options
- We are going to assemble our own workflow in this module.

---

## Initial Prompt Draft

- Step 1 -> Draft a prompt
- Step 2 -> Eval Dataset
- Step 3 -> Feed through Claude and get actual responses
- Step 4 -> Grade the responses by feed into a **Grader** that will grade and give scores and average them to get and objective way to eval.
- Step 5 -> Change Prompt and Repeat until get a sufficient score.

---

### 1 - Prompts

#### Goal
- write a prompt that will assist users in writing Python code, JSON config, or Regular Expressions focused on AWS-specific use cases

- Input: User will request code for a specific task

- Output: Python, JSON, or a regular expression without any explanation

Prompt v1
```python
  prompt = f"""
  Please provide a solution to the following task:

  {task}
  """
```

---

### 2 - Eval Datasets

- Each object contains a "task" that will merge into the prompt

- Can be assemble by hand or with Claude (if so, consider Haiku)

```python
[
  {
    "task": "Create a Python function to extract the AWS account ID from an ARN"
  },
  {
    "task": "Write a JSON policy document that allows read-only access to a specific S3 bucket"
  },
  ...
]
```

---

## Code

##### Create an API client and Helper Functions

In [45]:
from anthropic import Anthropic
from dotenv import load_dotenv
import os

load_dotenv()

token = os.environ["ANTHROPIC_AUTH_TOKEN"]
url = os.environ["ANTHROPIC_BASE_URL"]
model = os.environ["ANTHROPIC_DEFAULT_HAIKU_MODEL"]

client = Anthropic(
    api_key=token,
    base_url=url,
    default_headers={"Authorization": f"Bearer {token}"}
)

def add_user_message(messages, content):
    user_message = { "role": "user", "content": content }
    messages.append(user_message)

def add_assistant_message(messages, content):
    assistant_message = { "role": "assistant", "content": content }
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):

  params = {
    "model": model,
    "max_tokens": 1000,
    "messages": messages,
    "temperature": temperature,
    "stop_sequences": stop_sequences
  }

  if system:
    params["system"] = system

  message = client.messages.create(**params)
  return message.content[0].text

Define a function to generate a dataset, using a prompt that describes the task and gives output example.

In [46]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    response = chat(messages, stop_sequences=["```"])
    return json.loads(response)

Generate a dataset and save it to a file called: _dataset.json_

In [47]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

More helper funcionts

In [48]:
def run_prompt (test_case):
  """Merges the prompt and test case input, then returns the result"""
  prompt = f"""
  Please solve the following task.

  {test_case["task"]}
  """
  messages = []
  add_user_message(messages, prompt)
  output = chat(messages)
  return output

In [49]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # TODO - Grading
    score = 10

    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [50]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case for each test case"""
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    return results

Testing the Pipeline

In [51]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [52]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Region Extractor\n\nHere's a Python function that extracts the AWS region from an S3 bucket URL:\n\n```python\nimport re\n\ndef extract_region_from_s3_url(url: str) -> str:\n    \"\"\"\n    Extracts the AWS region from an S3 bucket URL.\n    \n    Args:\n        url (str): S3 bucket URL in format 's3://bucket-name.region.amazonaws.com'\n    \n    Returns:\n        str: The AWS region (e.g., 'us-east-1')\n    \n    Raises:\n        ValueError: If the URL format is invalid or region cannot be extracted\n    \"\"\"\n    # Pattern to match S3 URLs and extract region\n    pattern = r's3://[^.]+\\.([a-z0-9\\-]+)\\.amazonaws\\.com'\n    \n    match = re.search(pattern, url)\n    \n    if match:\n        return match.group(1)\n    else:\n        raise ValueError(f\"Invalid S3 URL format: {url}\")\n\n\n# Test cases\nif __name__ == \"__main__\":\n    # Test valid URLs\n    test_urls = [\n        \"s3://my-bucket.us-east-1.amazonaws.com\",\n        \"s3://data-bucket

---

### Graders

There are 3 different kind of Graders:

- Code
  - Programmatically evaluate the result
  - Useful for:
    - Checking output length
    - Verifying output does / doesn't have certain words
    - Syntax validation
    - Readability scores
- Model
  - Ask a model to assign a score to the output, or compare two versions
  - Useful for:
    - Response quality
    - Quality of instruction following
    - Completeness
    - Helpfulness
    - Safety
- Human
  - Ask a human to assign a score to the output, or compare two versions
  - Useful for:
    - General response quality
    - Comprehensiveness
    - Depth
    - Conciseness
    - Relevance

The only real requirements for the Graders is that they return a measure we can use, generally a number between 0-10

---

#### Evaluation Criteria - For our task

How will we know if the prompt is producing good outputs?

**Format** : Should return only Python, JSON, or Regex without explanation

**Valid Syntax**: Produced Python, JSON and Regex should have valid syntax

**Task Following**: Response should directly and clearly address the user's task. Generated code should be accurate.

<br>

We will try to validate the first two with a _Code Grader_, and the last one with a _Model Grader_ .

---

Back to the Code.

In [53]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [54]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # TODO - Grading
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [55]:
from statistics import mean

# Update the run_eval function to calculate the average score across all test cases
def run_eval(dataset):
    """Loads the dataset and calls run_test_case for each test case"""
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean(result["score"] for result in results)
    
    print(f"Average Score: {average_score:.2f}")

    return results

In [56]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average Score: 7.67


In [57]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# Solution\n\nHere's a Python function that extracts the AWS region from an S3 bucket URL:\n\n```python\nimport re\n\ndef extract_region_from_s3_url(url):\n    \"\"\"\n    Extracts the AWS region from an S3 bucket URL.\n    \n    Expected format: s3://bucket-name.region.amazonaws.com\n    \n    Args:\n        url (str): The S3 bucket URL\n        \n    Returns:\n        str: The AWS region, or None if the format is invalid\n    \"\"\"\n    # Pattern to match s3://bucket-name.region.amazonaws.com\n    pattern = r's3://[a-z0-9\\-]+\\.([a-z0-9\\-]+)\\.amazonaws\\.com'\n    match = re.search(pattern, url)\n    \n    if match:\n        return match.group(1)\n    return None\n\n\n# Test cases\nif __name__ == \"__main__\":\n    # Test with various S3 URLs\n    test_urls = [\n        \"s3://my-bucket.us-east-1.amazonaws.com\",\n        \"s3://data-bucket.eu-west-1.amazonaws.com\",\n        \"s3://archive.ap-southeast-2.amazonaws.com\",\n        \"s3://test-bucket.ca-centra

### Code Based Grading

Now we're going to define new functions to implement the validations using code (Validate JSON, Validate Python and Validate Regex)

In [58]:
import re
import ast

def validate_json(json_string):
    """Validates if a string is a valid JSON"""
    try:
        json.loads(json_string)
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(code_string):
    """Validates if a string is valid Python code"""
    try:
        ast.parse(code_string.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(regex_string):
    """Validates if a string is a valid regular expression"""
    try:
        re.compile(regex_string.strip())
        return 10
    except re.error:
        return 0

def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    elif format == "regex":
        return validate_regex(response)
    else:
        raise ValueError(f"Unknown format: {format}")

For the validate functions to work, we need some steps to improve our prompt from before and guarantee we have the type of generated content inside the dataset. Steps:

- Add functions to validate JSON / Python / Regex (Done)
- Make sure our dataset test cases indicate the type of generated content (JSON, Python or Regex)
- Update our draft prompt to make it clear we only want the relevant code
- Merge the scores from the model grader and the code grader

In [59]:
# Step 2: Function to generate a new dataset WITH THE FORMAT FIELD INCLUDED
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [60]:
# Generate the dataset and write it to 'dataset-with-format.json' - Same as before, this time the datase should have a format fied.
dataset = generate_dataset()
with open("dataset-with-format.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [61]:
# Step 3: Passes a test case into Claude - With updated prompt to make clear we only want relevant code.
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

In [62]:
# Step 4 - Added syntax_score and merge with model_score and calculate the average score
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [64]:
# To validate all changes, run again the evaluation and print the results
with open("dataset-with-format.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average Score: 7.50


---

### Is it good Enough?

- The only way to know is if we try to change our prompt in some way and get a better score.
- That's the exercise for the lesson.